# Clasificación de tono con Sonnet (Batch API) — reanudable

Toma los fragmentos semánticamente relevantes para cada grupo (misma lógica de `Analisis_Embeddings.ipynb`) y le pide a **Claude Sonnet**, vía la **Batch API**, que clasifique el tono de cada mención en una escala de 5 niveles.

Genera `clasificacion_menciones.csv` con las columnas:

```
fecha,mes,presidente,grupo,similitud,fragmento,valor,etiqueta
```

**Diseño reanudable — si se te acaban los créditos a la mitad:**
- El progreso se guarda en el CSV **lote por lote**, no hasta el final.
- Los lotes que ya se enviaron a Anthropic quedan registrados en `clasificacion_lotes_pendientes.json` — si el notebook se detiene, no se vuelven a enviar (evita pagar dos veces por lo mismo).
- Al volver a correr el notebook (después de recargar créditos), automáticamente detecta qué fragmentos ya están clasificados, cuáles siguen en proceso, y cuáles faltan por enviar — **no necesitas cambiar nada a mano**, solo vuelve a correr las celdas de la sección de clasificación.

> ⚠️ **Requiere una API key de Anthropic** como variable de entorno

---
## CONFIGURACIÓN

In [3]:
import os
from pathlib import Path

# =============================================================
# CONFIGURACIÓN
# =============================================================

# Cargar la clave de API de Claude desde un archivo local
_api_key_path = Path("api_key_claude.txt")
if not _api_key_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {_api_key_path}")

ANTHROPIC_API_KEY = _api_key_path.read_text(encoding="utf-8").strip()
if not ANTHROPIC_API_KEY:
    raise ValueError("El archivo api_key-claude.txt está vacío.")

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# Carpeta local donde están guardados tus archivos .docx
RUTA_DATOS = "./data/mananeras"

# Rango de fechas del análisis
FECHA_INICIO = "2018-12-01"   # formato: AAAA-MM-DD
FECHA_FIN    = "2026-01-31"   # formato: AAAA-MM-DD

# Corte entre presidencias
FECHA_CAMBIO_PRESIDENTE = "2024-10-01"

# =============================================================
# GRUPOS Y FRASES DESCRIPTIVAS PARA EMBEDDINGS SEMÁNTICOS
# =============================================================
GRUPOS_EMBEDDINGS = {
    "Periodistas": [
        "periodistas y prensa libre en México",
        "medios de comunicación y reporteros",
        "libertad de prensa y protección a periodistas",
        "comunicadores, corresponsales y periodistas de investigación",
        "agresiones a periodistas y libertad de expresión"
    ],

    "Ambientalistas": [
        "ambientalistas y ecologistas mexicanos",
        "protección del medio ambiente y sustentabilidad",
        "cambio climático y activismo ambiental",
        "defensa de los ecosistemas y la biodiversidad",
        "defensores del territorio y recursos naturales",
        "contaminación y mal manejo de residuos"
    ],

    "Colectivos_victimas": [
        "colectivos de víctimas y familiares de desaparecidos",
        "madres buscadoras y personas desaparecidas",
        "búsqueda de personas desaparecidas y fosas clandestinas",
        "verdad, justicia y reparación para las víctimas",
        "grupos afectados por la violencia"
    ],

    "ONGs": [
        "organizaciones de la sociedad civil y ONG",
        "asociaciones civiles sin fines de lucro",
        "organizaciones defensoras de derechos humanos",
        "organizaciones no gubernamentales y participación ciudadana",
        "sociedad civil organizada y organizaciones civiles"
    ],
}

# Umbral de similitud coseno para considerar un fragmento "relevante"
UMBRAL_SIMILITUD = 0.6  # Calibrado en Embeddings_Experimentacion.ipynb

# =============================================================
# CLASIFICACIÓN DE TONO CON SONNET (BATCH API)
# =============================================================
MODELO_CLASIFICACION = "claude-sonnet-5"

# Máximo de requests por lote enviado a la Batch API
TAMANO_MAX_LOTE = 2000

# Tokens máximos de salida por clasificación (la respuesta es un JSON muy corto)
MAX_TOKENS_SALIDA = 60

# Segundos de espera entre cada consulta de estado del lote
SEGUNDOS_ENTRE_CONSULTAS = 30

# Archivo CSV de salida (formato final pedido) y archivo de estado de lotes en vuelo
ARCHIVO_SALIDA_CSV   = "clasificacion_menciones.csv"
ARCHIVO_ESTADO_LOTES = "clasificacion_lotes_pendientes.json"

print("✅ Config guardada.")
print(f"   Grupos: {list(GRUPOS_EMBEDDINGS.keys())}")
print(f"   Umbral semántico: {UMBRAL_SIMILITUD}")
print(f"   Modelo de clasificación: {MODELO_CLASIFICACION}")


✅ Config guardada.
   Grupos: ['Periodistas', 'Ambientalistas', 'Colectivos_victimas', 'ONGs']
   Umbral semántico: 0.6
   Modelo de clasificación: claude-sonnet-5


---
## CÓDIGO BASE — No modificar

### Carga de archivos locales (.docx)

In [4]:
import os, re, json, time, hashlib
from datetime import datetime
from docx import Document
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Parsear fechas de configuración
_fecha_inicio = datetime.strptime(FECHA_INICIO, "%Y-%m-%d")
_fecha_fin    = datetime.strptime(FECHA_FIN,    "%Y-%m-%d")
_fecha_cambio = datetime.strptime(FECHA_CAMBIO_PRESIDENTE, "%Y-%m-%d")

_MESES_ES = {
    "ENE": 1, "FEB": 2, "MAR": 3, "ABR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AGO": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DIC": 12
}

def _extraer_fecha(nombre):
    m = re.search(r'([A-Za-z]{3})[-_](\d{4})', nombre)
    if m:
        mes_str = m.group(1).upper()
        año = int(m.group(2))
        mes = _MESES_ES.get(mes_str)
        if mes:
            return datetime(año, mes, 1)
    return None

def _cargar_corpus():
    if not os.path.exists(RUTA_DATOS):
        print(f"⚠️  Carpeta no encontrada: {RUTA_DATOS}")
        print("   Verifica que RUTA_DATOS apunte a la carpeta correcta en tu computadora.")
        return []
    archivos = []
    sin_fecha = []
    for nombre in sorted(os.listdir(RUTA_DATOS)):
        if not nombre.lower().endswith('.docx'):
            continue
        fecha = _extraer_fecha(nombre)
        if not fecha:
            sin_fecha.append(nombre)
            continue
        if not (_fecha_inicio <= fecha <= _fecha_fin):
            continue
        ruta = os.path.join(RUTA_DATOS, nombre)
        try:
            doc = Document(ruta)
            texto = ' '.join(p.text.strip() for p in doc.paragraphs if p.text.strip())
            presidente = "AMLO" if (fecha.year, fecha.month) < (_fecha_cambio.year, _fecha_cambio.month) else "Claudia Sheinbaum"
            archivos.append({
                'fecha':      fecha,
                'mes':        fecha.strftime('%Y-%m'),
                'año':        fecha.year,
                'presidente': presidente,
                'texto':      texto,
                'n_palabras': len(texto.split())
            })
        except Exception as e:
            print(f"   ⚠️  Error leyendo {nombre}: {e}")

    print(f"✅ {len(archivos)} archivos cargados")
    print(f"   Período: {_fecha_inicio.strftime('%b %Y')} → {_fecha_fin.strftime('%b %Y')}")
    dist = pd.Series([d['presidente'] for d in archivos]).value_counts()
    for p, n in dist.items():
        print(f"   {p}: {n} archivos")
    if sin_fecha:
        print(f"\n   ⚠️  {len(sin_fecha)} archivos ignorados (no se pudo leer fecha del nombre):")
        for f in sin_fecha[:5]:
            print(f"      {f}")
    return sorted(archivos, key=lambda x: x['fecha'])

_corpus = _cargar_corpus()


✅ 86 archivos cargados
   Período: Dec 2018 → Jan 2026
   AMLO: 70 archivos
   Claudia Sheinbaum: 16 archivos


---
### Embeddings semánticos y extracción de fragmentos relevantes

Cada fragmento recibe un **`id_fragmento`**: un hash determinístico de `(fecha, grupo, fragmento)`.
Es la clave que usamos para reconocer "ya clasifiqué esto" aunque se reinicie el notebook — no depende
de la posición en la lista, depende del contenido en sí.

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def _generar_id_fragmento(fecha, grupo, fragmento):
    """Hash determinístico -- el mismo fragmento siempre produce el mismo id, sin importar el orden."""
    fecha_str = pd.to_datetime(fecha).strftime('%Y-%m-%d')
    base = f"{fecha_str}|{grupo}|{fragmento}"
    return hashlib.md5(base.encode('utf-8')).hexdigest()[:16]

_fragmentos_relevantes = []

if not _corpus:
    print("No hay archivos cargados. Revisa RUTA_DATOS y vuelve a ejecutar la celda de carga.")
else:
    print("Cargando modelo de embeddings (paraphrase-multilingual-MiniLM-L12-v2)...")
    _modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print("✅ Modelo listo.\n")

    print("Generando vectores de grupos de interés...")
    _emb_grupos = {}
    for grupo, frases in GRUPOS_EMBEDDINGS.items():
        vecs = _modelo.encode(frases, show_progress_bar=False)
        _emb_grupos[grupo] = np.mean(vecs, axis=0)
        print(f"   ✓ {grupo}")

    def _chunking(texto, n=3):
        oraciones = [s.strip() for s in re.split(r'(?<=[.!?])\s+', texto) if len(s.strip()) > 40]
        return [' '.join(oraciones[i:i+n]) for i in range(0, len(oraciones), n)
                if len(' '.join(oraciones[i:i+n])) > 60]

    print(f"\nProcesando {len(_corpus)} archivos...")
    print(f"Umbral de similitud: {UMBRAL_SIMILITUD}\n")

    for i, doc in enumerate(_corpus):
        if (i+1) % 30 == 0 or i == 0:
            print(f"   Archivo {i+1}/{len(_corpus)} — {doc['fecha'].strftime('%b %Y')} ({doc['presidente']})")
        chunks = _chunking(doc['texto'])
        if not chunks:
            continue
        emb_chunks = _modelo.encode(chunks, batch_size=64, show_progress_bar=False)

        for grupo, emb_g in _emb_grupos.items():
            sims = cosine_similarity([emb_g], emb_chunks)[0]
            for idx_chunk, sim in enumerate(sims):
                if sim >= UMBRAL_SIMILITUD:
                    fragmento_texto = chunks[idx_chunk]
                    _fragmentos_relevantes.append({
                        'fecha':        doc['fecha'],
                        'mes':          doc['mes'],
                        'presidente':   doc['presidente'],
                        'grupo':        grupo,
                        'similitud':    round(float(sim), 3),
                        'fragmento':    fragmento_texto,
                        'id_fragmento': _generar_id_fragmento(doc['fecha'], grupo, fragmento_texto)
                    })

    df_frag = pd.DataFrame(_fragmentos_relevantes)
    print(f"\n✅ {len(df_frag)} fragmentos relevantes encontrados en total.")
    if not df_frag.empty:
        print(df_frag.groupby('grupo').size().to_string())


Cargando modelo de embeddings (paraphrase-multilingual-MiniLM-L12-v2)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6122.89it/s]


✅ Modelo listo.

Generando vectores de grupos de interés...
   ✓ Periodistas
   ✓ Ambientalistas
   ✓ Colectivos_victimas
   ✓ ONGs

Procesando 86 archivos...
Umbral de similitud: 0.6

   Archivo 1/86 — Dec 2018 (AMLO)
   Archivo 30/86 — May 2021 (AMLO)
   Archivo 60/86 — Nov 2023 (AMLO)

✅ 3057 fragmentos relevantes encontrados en total.
grupo
Ambientalistas          552
Colectivos_victimas     235
ONGs                    110
Periodistas            2160


---
## Clasificación de tono con Sonnet (Batch API)

Escala usada:

| Valor | Etiqueta | Descripción |
|---|---|---|
| -2 | Ataque directo | Insulto o acusación grave contra el grupo |
| -1 | Crítica | Cuestionamiento de su labor o conducta, sin insulto |
| 0 | Neutral | Mención factual, sin carga valorativa |
| +1 | Reconocimiento | Elogio puntual o agradecimiento |
| +2 | Defensa/respaldo | Defensa explícita del grupo o su labor |

Las siguientes celdas definen las funciones que hacen posible reanudar el proceso:
- `_cargar_ya_clasificados()` — lee `ARCHIVO_SALIDA_CSV` y regenera el `id_fragmento` de cada fila ya guardada, para saber qué NO hay que volver a enviar.
- `_cargar_lotes_pendientes()` / `_guardar_lotes_pendientes()` — llevan el registro de lotes ya enviados a Anthropic que aún no se han recuperado.
- `_procesar_resultados_lote()` — descarga los resultados de un lote terminado y los agrega al CSV (append, no sobreescribe).
- `_resolver_lotes_pendientes()` — revisa todos los lotes pendientes; si `esperar=True`, hace polling hasta que todos terminen.

In [18]:
import anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "No se encontró la variable de entorno ANTHROPIC_API_KEY. "
    )

client = anthropic.Anthropic()

RUBRICA = (
    "Eres un clasificador de tono para análisis de discurso político en México. "
    "Se te da un fragmento de una conferencia de prensa (mañanera) y el nombre de un grupo social. "
    "Clasifica cómo se presenta a ese grupo en el fragmento, usando esta escala:\n\n"
    "-2 Ataque directo: insulto o acusación grave contra el grupo\n"
    "-1 Crítica: cuestionamiento de su labor o conducta, sin insulto\n"
    " 0 Neutral: mención factual, sin carga valorativa\n"
    "+1 Reconocimiento: elogio puntual o agradecimiento\n"
    "+2 Defensa/respaldo: defensa explícita del grupo o su labor\n\n"
    "Responde ÚNICAMENTE con un objeto JSON, sin texto adicional, con este formato exacto:\n"
    '{"valor": <entero de -2 a 2>, "etiqueta": "<Ataque directo|Crítica|Neutral|Reconocimiento|Defensa>"}'
)

COLUMNAS_FINALES = ['fecha', 'mes', 'presidente', 'grupo', 'similitud', 'fragmento', 'valor', 'etiqueta']

def _extraer_texto_respuesta(content_blocks):
    """Busca el primer bloque de tipo 'text' -- ignora bloques de 'thinking' u otros
    que puedan venir antes en la respuesta."""
    for block in content_blocks:
        if getattr(block, "type", None) == "text":
            return block.text
    return ""


def _parsear_respuesta(texto_respuesta):
    m = re.search(r'\{.*\}', texto_respuesta, re.DOTALL)
    if not m:
        return None, None
    try:
        data = json.loads(m.group(0))
        return data.get("valor"), data.get("etiqueta")
    except (json.JSONDecodeError, AttributeError):
        return None, None


def _cargar_ya_clasificados():
    """Regenera el id_fragmento de cada fila ya guardada en el CSV final, para no reenviarla."""
    if not os.path.exists(ARCHIVO_SALIDA_CSV):
        return set()
    prev = pd.read_csv(ARCHIVO_SALIDA_CSV, encoding='utf-8-sig')
    if prev.empty:
        return set()
    ids = prev.apply(lambda r: _generar_id_fragmento(r['fecha'], r['grupo'], r['fragmento']), axis=1)
    return set(ids)


def _cargar_lotes_pendientes():
    if os.path.exists(ARCHIVO_ESTADO_LOTES):
        with open(ARCHIVO_ESTADO_LOTES, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []


def _guardar_lotes_pendientes(lotes):
    with open(ARCHIVO_ESTADO_LOTES, 'w', encoding='utf-8') as f:
        json.dump(lotes, f, ensure_ascii=False, indent=2)


def _append_resultados_csv(df_resultados):
    existe = os.path.exists(ARCHIVO_SALIDA_CSV)
    df_resultados[COLUMNAS_FINALES].to_csv(
        ARCHIVO_SALIDA_CSV, mode='a', header=not existe, index=False, encoding='utf-8-sig'
    )


def _procesar_resultados_lote(batch_id, ids_fragmentos_esperados):
    """Descarga los resultados de un lote YA terminado y los agrega (append) al CSV final."""
    filas_resultado = []
    ids_recibidos = set()
    for entry in client.messages.batches.results(batch_id):
        id_frag = entry.custom_id
        ids_recibidos.add(id_frag)
        fila_original = df_frag[df_frag['id_fragmento'] == id_frag]
        if fila_original.empty:
            continue
        fila_original = fila_original.iloc[0]
        if entry.result.type == "succeeded":
            texto = _extraer_texto_respuesta(entry.result.message.content)
            valor, etiqueta = _parsear_respuesta(texto)
        else:
            valor, etiqueta = None, f"error:{entry.result.type}"
        filas_resultado.append({
            'fecha': fila_original['fecha'], 'mes': fila_original['mes'],
            'presidente': fila_original['presidente'], 'grupo': fila_original['grupo'],
            'similitud': fila_original['similitud'], 'fragmento': fila_original['fragmento'],
            'valor': valor, 'etiqueta': etiqueta
        })

    faltantes = set(ids_fragmentos_esperados) - ids_recibidos
    if faltantes:
        print(f"   ⚠️  {len(faltantes)} fragmentos del lote {batch_id} no aparecieron en los resultados.")

    if filas_resultado:
        _append_resultados_csv(pd.DataFrame(filas_resultado))
        print(f"   ✅ {len(filas_resultado)} resultados del lote {batch_id} guardados en {ARCHIVO_SALIDA_CSV}")


def _resolver_lotes_pendientes(esperar=False):
    """Revisa los lotes pendientes; si esperar=True, hace polling hasta que todos terminen."""
    while True:
        lotes = _cargar_lotes_pendientes()
        if not lotes:
            print("No quedan lotes pendientes.")
            return
        aun_pendientes = []
        for lote in lotes:
            estado = client.messages.batches.retrieve(lote['batch_id'])
            conteos = estado.request_counts
            print(f"   {lote['batch_id']}: {estado.processing_status} "
                  f"(procesando={conteos.processing}, listos={conteos.succeeded}, "
                  f"errores={conteos.errored})")
            if estado.processing_status == "ended":
                _procesar_resultados_lote(lote['batch_id'], lote['ids_fragmentos'])
            else:
                aun_pendientes.append(lote)
        _guardar_lotes_pendientes(aun_pendientes)
        if not aun_pendientes or not esperar:
            return
        print(f"   Esperando {SEGUNDOS_ENTRE_CONSULTAS}s antes de volver a consultar...\n")
        time.sleep(SEGUNDOS_ENTRE_CONSULTAS)


def _construir_requests(df):
    requests = []
    for _, fila in df.iterrows():
        contenido_usuario = f"Grupo: {fila['grupo']}\nFragmento: \"{fila['fragmento']}\""
        requests.append(
            Request(
                custom_id=fila['id_fragmento'],
                params=MessageCreateParamsNonStreaming(
                    model=MODELO_CLASIFICACION,
                    max_tokens=MAX_TOKENS_SALIDA,
                    system=[{"type": "text", "text": RUBRICA, "cache_control": {"type": "ephemeral"}}],
                    messages=[{"role": "user", "content": contenido_usuario}]
                )
            )
        )
    return requests

print("✅ Funciones de clasificación y reanudación listas.")


✅ Funciones de clasificación y reanudación listas.


### Paso 1 — Resolver lotes pendientes de una corrida anterior (si los hay)

Si el notebook se detuvo antes (por ejemplo, por falta de créditos), aquí se recuperan los lotes que ya habían
terminado de procesarse en Anthropic mientras el notebook no estaba corriendo — sin volver a pagar por ellos.

In [17]:
_resolver_lotes_pendientes(esperar=False)


   msgbatch_011q28f9xJQQKvhbPT6VRjN7: ended (procesando=0, listos=2000, errores=0)


AttributeError: 'ThinkingBlock' object has no attribute 'text'

### Paso 2 — Enviar los fragmentos que aún faltan

Calcula qué fragmentos **ya están clasificados** (en el CSV) y cuáles **ya están en proceso** (en un lote pendiente),
y solo envía los que de verdad faltan. Si se agotan los créditos a la mitad, los lotes ya enviados quedan
guardados en `ARCHIVO_ESTADO_LOTES` — puedes volver a correr esta celda después de recargar saldo y va a
seguir exactamente donde se quedó.

In [10]:
ya_clasificados = _cargar_ya_clasificados()
ids_en_lotes_pendientes = {fid for lote in _cargar_lotes_pendientes() for fid in lote['ids_fragmentos']}

pendientes_de_envio = df_frag[
    ~df_frag['id_fragmento'].isin(ya_clasificados) &
    ~df_frag['id_fragmento'].isin(ids_en_lotes_pendientes)
]

print(f"{len(ya_clasificados)} fragmentos ya clasificados en corridas anteriores.")
print(f"{len(ids_en_lotes_pendientes)} fragmentos en lotes que ya están en proceso.")
print(f"{len(pendientes_de_envio)} fragmentos nuevos por enviar.\n")

if pendientes_de_envio.empty:
    print("No hay fragmentos nuevos que enviar — todo ya está clasificado o en proceso.")
else:
    indices = list(pendientes_de_envio.index)
    partes = [indices[i:i + TAMANO_MAX_LOTE] for i in range(0, len(indices), TAMANO_MAX_LOTE)]
    print(f"Enviando en {len(partes)} lote(s) nuevo(s)...")

    credito_agotado = False
    for n_lote, idxs in enumerate(partes):
        if credito_agotado:
            print(f"   Lote {n_lote+1}/{len(partes)} NO enviado (créditos agotados). "
                  f"Vuelve a correr esta celda tras recargar saldo.")
            continue

        sub_df = pendientes_de_envio.loc[idxs]
        requests = _construir_requests(sub_df)
        try:
            batch = client.messages.batches.create(requests=requests)
        except anthropic.BadRequestError as e:
            mensaje = str(e)
            if "credit balance" in mensaje.lower():
                print(f"   ❌ Te quedaste sin créditos al enviar el lote {n_lote+1}/{len(partes)}.")
                print(f"      Los {n_lote} lote(s) anteriores YA se enviaron y NO se pierden.")
                print("      Recarga créditos en https://console.anthropic.com/settings/billing")
                print("      y vuelve a correr esta celda — los fragmentos ya enviados o clasificados se saltan solos.")
                credito_agotado = True
                continue
            else:
                raise

        # Guardar el estado de inmediato — si el notebook se cae después de esto, no se pierde el registro.
        nuevo_lote = {'batch_id': batch.id, 'ids_fragmentos': list(sub_df['id_fragmento'])}
        _guardar_lotes_pendientes(_cargar_lotes_pendientes() + [nuevo_lote])
        print(f"   Lote {n_lote+1}/{len(partes)} enviado — batch_id: {batch.id} ({len(sub_df)} fragmentos)")


0 fragmentos ya clasificados en corridas anteriores.
0 fragmentos en lotes que ya están en proceso.
3057 fragmentos nuevos por enviar.

Enviando en 2 lote(s) nuevo(s)...
   Lote 1/2 enviado — batch_id: msgbatch_011q28f9xJQQKvhbPT6VRjN7 (2000 fragmentos)
   Lote 2/2 enviado — batch_id: msgbatch_01MQYAZCi6ZDZR6uhFvabdsi (1057 fragmentos)


### Paso 3 — Esperar a que todo termine

Esta celda hace polling (revisa cada `SEGUNDOS_ENTRE_CONSULTAS`) hasta que todos los lotes —los de antes y los recién
enviados— terminen, y va guardando cada resultado en el CSV apenas está listo.

In [21]:
_resolver_lotes_pendientes(esperar=True)


   msgbatch_011q28f9xJQQKvhbPT6VRjN7: ended (procesando=0, listos=2000, errores=0)
   ✅ 2000 resultados del lote msgbatch_011q28f9xJQQKvhbPT6VRjN7 guardados en clasificacion_menciones.csv
   msgbatch_01MQYAZCi6ZDZR6uhFvabdsi: ended (procesando=0, listos=1057, errores=0)
   ✅ 1057 resultados del lote msgbatch_01MQYAZCi6ZDZR6uhFvabdsi guardados en clasificacion_menciones.csv


### Resumen final

Lee el CSV directamente del disco (no de memoria), así siempre refleja el estado real guardado,
sin importar cuántas veces se haya reanudado el proceso.

In [22]:
df_final = pd.read_csv(ARCHIVO_SALIDA_CSV, encoding='utf-8-sig')
print(f"✅ {ARCHIVO_SALIDA_CSV} contiene {len(df_final)} menciones clasificadas.")
print(df_final.groupby('grupo').size().to_string())
print()
print(df_final['etiqueta'].value_counts().to_string())
df_final.head(10)


✅ clasificacion_menciones.csv contiene 3057 menciones clasificadas.
grupo
Ambientalistas          552
Colectivos_victimas     235
ONGs                    110
Periodistas            2160

etiqueta
Neutral           1092
Crítica            784
Ataque directo     550
Reconocimiento     411
Defensa            216


,fecha,mes,presidente,grupo,similitud,fragmento,valor,etiqueta
0,2022-10-01,2022-10,AMLO,Periodistas,0.690,"Comisión de la Verdad Ayotzinapa, es lo mismo ...",0.0,Neutral
1,2019-02-01,2019-02,AMLO,ONGs,0.626,"Entonces, por eso es importante, no siempre ha...",-1.0,Crítica
2,2020-04-01,2020-04,AMLO,Periodistas,0.632,"Hans Salazar, de ZMG Noticias, Gurú Político y...",0.0,Neutral
3,2023-03-01,2023-03,AMLO,Periodistas,0.727,"Hay algunos que todavía, ¿no?, sirven de tapad...",-1.0,Crítica
4,2022-02-01,2022-02,AMLO,Periodistas,0.694,"Y, desde luego, todo nuestro respeto al noble ...",2.0,Defensa
5,2019-06-01,2019-06,AMLO,Periodistas,0.740,"Entonces, bueno, a eso me refería, que muchos ...",-1.0,Crítica
6,2023-04-01,2023-04,AMLO,Periodistas,0.605,"Porque, imagínense, si fuésemos corruptos, pue...",-2.0,Ataque directo
7,2020-06-01,2020-06,AMLO,ONGs,0.611,La UNAM tiene 450 mil miembros en su comunidad...,0.0,Neutral
8,2023-06-01,2023-06,AMLO,Ambientalistas,0.616,Reafirmamos el compromiso de resguardar el pat...,0.0,Neutral
9,2022-02-01,2022-02,AMLO,Periodistas,0.634,"Presidente, me solidarizo con las familias de ...",-1.0,Crítica
